<a href="https://colab.research.google.com/github/sun-mengwei/dtb-colab-experiments/blob/codex%2Fgame-dynamics-dtb/DTB_Game_Ver2/cournot_10d_nonpotential_mlp_dtb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ten-dimensional non-potential Cournot game: frozen-θ MLP–DTB stress test

This notebook generalizes the supplied three-player opponent-total Cournot game to a configurable high-dimensional game, with `DIM=10` by default. It is designed to probe where the direct DTB heuristic becomes inaccurate or expensive, rather than merely produce an attractive two-dimensional projection.

For player \(i\), let

\[
r_i(x)=\sum_{j\ne i}x_j,
\qquad
\operatorname{BR}_i(x)=\max\{\mu r_i(1-r_i),0\},
\]

and use the constrained best-response velocity

\[
b_i(x)=2b\,[\operatorname{BR}_i(x)-x_i].
\]

The `max` enforces the nonnegative quantity constraint. Without it, the literal 10D extension produces large negative best responses whenever the opponents' total exceeds one, and the experiment mainly measures escape into economically meaningless negative quantities. Set `CONSTRAINED_BEST_RESPONSE=False` to test that raw extension deliberately.

This experiment updates the physical particles only. The MLP parameters remain fixed at \(\theta_0\), so its tangent basis can be constructed and factorized once. Given fixed labels \(z_i\) and physical particles \(X_{k-1}(z_i)\), solve

\[
\alpha_k=\arg\min_\alpha\sum_i
\left\|D_\theta T_{\theta_0}(z_i)\alpha-b(X_{k-1}(z_i))\right\|^2,
\]

then update

\[
X_k=X_{k-1}+hD_\theta T_{\theta_0}\alpha_k,
\qquad \theta_k\equiv\theta_0.
\]

Only the selected tangent coordinates are active; equivalently, \(\alpha_k\) is zero outside that subset. A later experiment can evolve \(\theta\), but this notebook deliberately does not. The fixed Jacobian and its truncated SVD are cached, so setup cost and per-step coefficient-solve cost are reported separately.

The review uses synchronized timing, a matched explicit-Euler reference, PCA with a fixed projection basis, pairwise coordinate views, parallel coordinates, covariance heatmaps, dominant-pair endpoint counts, direct endpoint equilibrium diagnostics, and an optional dimension-scaling benchmark. All DTB results come directly from the tangent-bundle trajectory at the configured final time. No two-dimensional view is treated as proof of high-dimensional convergence.

In [ ]:
# BLOCK 0 — Colab/repository setup and editable stress-test controls.
from pathlib import Path
from itertools import combinations
import os, subprocess, sys, math, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.func import jacrev
from IPython.display import display

REPO = Path("/content/dtb-colab-experiments")
BRANCH = "codex/game-dynamics-dtb"
if not (REPO / ".git").exists():
    subprocess.run([
        "git", "clone", "-q", "--depth", "1", "--branch", BRANCH,
        "https://github.com/sun-mengwei/dtb-colab-experiments.git", str(REPO)
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "-q", "--ff-only"], check=True)
EXPERIMENT_DIR = REPO / "DTB_Game_Ver2"
sys.path.insert(0, str(EXPERIMENT_DIR))
sys.path.insert(0, str(REPO / "DTB_Game_Ver1"))
os.chdir(EXPERIMENT_DIR)

try:
    import torch._dynamo.compiled_autograd
except (AttributeError, ImportError):
    pass
from dtb import device, flat_params
from run_game_dtb import ResidualMLPMap, game_dtb_basis_matrices, map_at
from game_dtb.projection import truncated_svd_solve

# -------------------------- main experiment --------------------------
SEED = 2026
DIM = 10                 # Increase this after the 10D baseline is understood.
N_PARTICLES = 384
T_FINAL, H = 1.0, 0.01
N_STEPS = round(T_FINAL / H)
MLP_WIDTH, MLP_DEPTH = 24, 2
BASIS_SIZE = 512         # Restricted tangent coordinates; output layer is forced in.
SVD_RTOL = 1e-3
JACOBIAN_CHUNK = 64
INITIAL_TOTAL = 1.0      # Dirichlet samples scaled to this total quantity.
COURNOT_B, COURNOT_MU = 1.0, 2.0
CONSTRAINED_BEST_RESPONSE = True
SNAPSHOT_STEPS = sorted(set([0, N_STEPS // 4, N_STEPS // 2,
                             3 * N_STEPS // 4, N_STEPS]))

# A one-step microbenchmark exposes dimension scaling without rerunning full trajectories.
RUN_SCALING_BENCHMARK = True
BENCHMARK_DIMS = (3, 5, 10, 15, 20)
BENCHMARK_PARTICLES = 64
BENCHMARK_BASIS_SIZE = 512

DEVICE = device()
DTYPE = torch.float32
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.set_float32_matmul_precision("high")
print({
    "device": str(DEVICE), "dimension": DIM, "particles": N_PARTICLES,
    "steps": N_STEPS, "h": H, "MLP": (MLP_WIDTH, MLP_DEPTH),
    "basis_size": BASIS_SIZE, "svd_rtol": SVD_RTOL,
    "constrained_best_response": CONSTRAINED_BEST_RESPONSE,
})

In [ ]:
# BLOCK 1 — Define the high-dimensional non-potential game and known checks.
def opponents_total(x):
    return x.sum(dim=-1, keepdim=True) - x

def cournot_velocity(x, constrained=CONSTRAINED_BEST_RESPONSE):
    """High-dimensional opponent-total Cournot best-response velocity."""
    r = opponents_total(x)
    best_response = COURNOT_MU * r * (1.0 - r)
    if constrained:
        best_response = torch.clamp_min(best_response, 0.0)
    return 2.0 * COURNOT_B * (best_response - x)

def analytic_equilibria(dim, device_=DEVICE, dtype_=DTYPE):
    """Origin, symmetric interior point, and all pair-support points for mu=2."""
    points = [torch.zeros(dim, device=device_, dtype=dtype_)]
    names = ["origin"]
    q = (COURNOT_MU * (dim - 1) - 1.0) / (COURNOT_MU * (dim - 1) ** 2)
    points.append(torch.full((dim,), q, device=device_, dtype=dtype_))
    names.append("symmetric")
    if abs(COURNOT_MU - 2.0) < 1e-12:
        for first, second in combinations(range(dim), 2):
            point = torch.zeros(dim, device=device_, dtype=dtype_)
            point[first] = point[second] = 0.5
            points.append(point)
            names.append(f"pair({first + 1},{second + 1})")
    return torch.stack(points), names

known_equilibria, known_equilibrium_names = analytic_equilibria(DIM)
known_residuals = torch.linalg.vector_norm(
    cournot_velocity(known_equilibria), dim=1
)
assert float(known_residuals.max()) < 2e-5

# A potential game requires a symmetric pseudo-gradient Jacobian. This one is not.
probe = torch.linspace(0.01, 0.10, DIM, device=DEVICE, dtype=DTYPE)
probe_jacobian = jacrev(cournot_velocity)(probe)
nonpotential_asymmetry = torch.linalg.norm(probe_jacobian - probe_jacobian.T)
assert float(nonpotential_asymmetry) > 1e-3

symmetric_value = float(known_equilibria[1, 0])
print({
    "known_equilibria": len(known_equilibrium_names),
    "pair_support_equilibria": math.comb(DIM, 2),
    "symmetric_coordinate": symmetric_value,
    "max_known_residual": float(known_residuals.max()),
    "nonpotential_J_minus_JT_norm": float(nonpotential_asymmetry),
})

In [ ]:
# BLOCK 2 — Initialize the identity MLP map and a high-value tangent subset.
def sample_simplex(count, dim, total=1.0, device_=DEVICE, dtype_=DTYPE):
    """Dirichlet(1,...,1) samples, represented without an extra dependency."""
    uniforms = torch.rand(count, dim, device=device_, dtype=dtype_).clamp_min(1e-7)
    exponentials = -torch.log(uniforms)
    return total * exponentials / exponentials.sum(dim=1, keepdim=True)

def choose_tangent_parameters(structure_, depth, budget, device_=DEVICE):
    """Include the entire output layer, then randomly fill the remaining budget."""
    output_prefix = f"net.{2 * depth}."
    forced, offset = [], 0
    for name, shape in structure_:
        count = math.prod(shape)
        if name.startswith(output_prefix):
            forced.append(torch.arange(offset, offset + count, device=device_))
        offset += count
    forced = torch.cat(forced)
    budget = min(int(budget), offset)
    if forced.numel() > budget:
        raise ValueError(
            f"basis budget {budget} is smaller than output-layer size {forced.numel()}"
        )
    mask = torch.ones(offset, dtype=torch.bool, device=device_)
    mask[forced] = False
    remaining = torch.arange(offset, device=device_)[mask]
    extra_count = budget - forced.numel()
    if extra_count:
        extra = remaining[torch.randperm(remaining.numel(), device=device_)[:extra_count]]
        selected_ = torch.cat((forced, extra))
    else:
        selected_ = forced
    return selected_.sort().values, int(forced.numel()), offset

labels = sample_simplex(N_PARTICLES, DIM, INITIAL_TOTAL)
model = ResidualMLPMap(
    dim=DIM, width=MLP_WIDTH, depth=MLP_DEPTH,
    activation="tanh", dtype=DTYPE
).to(DEVICE)
theta_0, structure, _ = flat_params(model)
theta_0 = theta_0.to(DEVICE)
selected, forced_output_parameters, total_parameters = choose_tangent_parameters(
    structure, MLP_DEPTH, BASIS_SIZE
)
x_0 = map_at(theta_0, labels, model, structure).detach()
assert torch.allclose(x_0, labels, atol=1e-7, rtol=0)

system_rows = N_PARTICLES * DIM
jacobian_megabytes = system_rows * selected.numel() * 4 / 2**20
print({
    "total_trainable_parameters": total_parameters,
    "selected_tangent_parameters": int(selected.numel()),
    "forced_output_parameters": forced_output_parameters,
    "stacked_system_shape": (system_rows, int(selected.numel())),
    "raw_J_storage_MB_fp32": round(jacobian_megabytes, 2),
})

In [ ]:
# BLOCK 3 — Particle-only DTB with frozen parameters and a cached tangent SVD.
def synchronize():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

def particle_rms(values):
    return torch.sqrt(torch.mean(torch.sum(values.square(), dim=-1)))

def factor_fixed_tangent(stacked_jacobian):
    u, singular_values, vh = torch.linalg.svd(
        stacked_jacobian, full_matrices=False
    )
    if singular_values.numel() == 0 or float(singular_values[0]) == 0.0:
        keep = torch.zeros_like(singular_values, dtype=torch.bool)
    else:
        keep = singular_values > SVD_RTOL * singular_values[0]
    return {
        "matrix": stacked_jacobian,
        "u": u[:, keep], "s": singular_values[keep], "vh": vh[keep],
        "rank": int(keep.sum()),
        "sigma_max": float(singular_values[0]) if singular_values.numel() else 0.0,
        "sigma_min": float(singular_values[keep][-1]) if bool(keep.any()) else 0.0,
    }

def solve_with_fixed_tangent(factor, stacked_velocity, parameter_count):
    if factor["rank"] == 0:
        alpha = torch.zeros(
            parameter_count, device=stacked_velocity.device, dtype=stacked_velocity.dtype
        )
    else:
        coefficients = (factor["u"].T @ stacked_velocity) / factor["s"]
        alpha = factor["vh"].T @ coefficients
    residual = factor["matrix"] @ alpha - stacked_velocity
    relative_residual = float(
        torch.linalg.vector_norm(residual)
        / (torch.linalg.vector_norm(stacked_velocity) + 1e-30)
    )
    return alpha, relative_residual

def run_high_dimensional_dtb(theta_initial, fixed_labels):
    theta_fixed = theta_initial.detach().clone()
    x_previous = map_at(theta_fixed, fixed_labels, model, structure).detach()
    history = {name: [] for name in (
        "trajectory", "projection_residual", "retained_rank",
        "sigma_max", "sigma_min_retained", "alpha_norm",
        "target_drift_rms", "projected_drift_rms",
        "negative_coordinate_fraction", "nearest_known_median",
        "nearest_known_p90", "solve_seconds", "update_seconds",
    )}
    history["trajectory"].append(x_previous.cpu())
    # theta is frozen, so J(theta_0, z) and its SVD are one-time setup costs.
    synchronize(); wall_start = time.perf_counter()
    _, jacobians, stacked_jacobian = game_dtb_basis_matrices(
        theta_fixed, selected, fixed_labels, model, structure,
        chunk=JACOBIAN_CHUNK
    )
    synchronize(); after_basis = time.perf_counter()
    factor = factor_fixed_tangent(stacked_jacobian)
    synchronize(); after_factor = time.perf_counter()
    history["basis_setup_seconds"] = after_basis - wall_start
    history["svd_setup_seconds"] = after_factor - after_basis
    integration_start = after_factor

    for step in range(1, N_STEPS + 1):
        target_velocity = cournot_velocity(x_previous)
        synchronize(); solve_start = time.perf_counter()
        alpha_k, relative_residual = solve_with_fixed_tangent(
            factor, target_velocity.reshape(-1), selected.numel()
        )
        synchronize(); after_solve = time.perf_counter()

        projected_velocity = torch.einsum("ndm,m->nd", jacobians, alpha_k)
        x_k = x_previous + H * projected_velocity.detach()
        nearest_distance = torch.cdist(x_k, known_equilibria).min(dim=1).values
        synchronize(); after_update = time.perf_counter()

        history["trajectory"].append(x_k.cpu())
        history["projection_residual"].append(relative_residual)
        history["retained_rank"].append(factor["rank"])
        history["sigma_max"].append(factor["sigma_max"])
        history["sigma_min_retained"].append(factor["sigma_min"])
        history["alpha_norm"].append(float(torch.linalg.vector_norm(alpha_k)))
        history["target_drift_rms"].append(float(particle_rms(target_velocity)))
        history["projected_drift_rms"].append(float(particle_rms(projected_velocity)))
        history["negative_coordinate_fraction"].append(float((x_k < 0).float().mean()))
        history["nearest_known_median"].append(float(nearest_distance.median()))
        history["nearest_known_p90"].append(float(torch.quantile(nearest_distance, 0.9)))
        history["solve_seconds"].append(after_solve - solve_start)
        history["update_seconds"].append(after_update - after_solve)

        x_previous = x_k.detach()
        if step % max(1, N_STEPS // 10) == 0:
            print(
                f"{step:4d}/{N_STEPS}: residual={relative_residual:.2e}, "
                f"rank={factor['rank']:3d}, drift-rms={history['target_drift_rms'][-1]:.2e}, "
                f"median-eq-distance={history['nearest_known_median'][-1]:.3e}"
            )

    synchronize()
    history["wall_seconds"] = time.perf_counter() - wall_start
    history["integration_seconds"] = time.perf_counter() - integration_start
    history["trajectory"] = torch.stack(history["trajectory"])
    for key in tuple(history):
        if isinstance(history[key], list):
            history[key] = np.asarray(history[key])
    return theta_fixed, history

In [ ]:
# BLOCK 4 — Run DTB and a matched explicit-Euler reference.
theta_final, result = run_high_dimensional_dtb(theta_0, labels)

synchronize(); euler_start = time.perf_counter()
x_reference = labels.detach().clone()
reference_history = [x_reference.cpu()]
for _ in range(N_STEPS):
    x_reference = x_reference + H * cournot_velocity(x_reference)
    reference_history.append(x_reference.cpu())
synchronize(); euler_seconds = time.perf_counter() - euler_start
reference_trajectory = torch.stack(reference_history)

assert result["trajectory"].shape == reference_trajectory.shape
assert torch.isfinite(result["trajectory"]).all() and torch.isfinite(theta_final).all()
assert torch.equal(theta_final, theta_0), "theta must remain frozen in this notebook"

trajectory_rmse = torch.sqrt(torch.mean(torch.sum(
    (result["trajectory"] - reference_trajectory).square(), dim=-1
), dim=1))
reference_rms = torch.sqrt(torch.mean(torch.sum(
    reference_trajectory.square(), dim=-1
), dim=1))
relative_rmse = trajectory_rmse / (reference_rms + 1e-12)
reference_drift_rms = torch.sqrt(torch.mean(torch.sum(
    cournot_velocity(reference_trajectory.to(DEVICE)).square(), dim=-1
), dim=1)).cpu()

timing_summary = pd.DataFrame([{
    "dimension": DIM,
    "particles": N_PARTICLES,
    "steps": N_STEPS,
    "selected_basis": int(selected.numel()),
    "DTB total wall seconds": result["wall_seconds"],
    "DTB integration seconds": result["integration_seconds"],
    "Euler wall seconds": euler_seconds,
    "one-time basis setup seconds": result["basis_setup_seconds"],
    "one-time SVD setup seconds": result["svd_setup_seconds"],
    "integration seconds/step": result["integration_seconds"] / N_STEPS,
    "median cached coefficient-solve seconds": np.median(result["solve_seconds"]),
    "final relative RMSE": float(relative_rmse[-1]),
    "final projection residual": result["projection_residual"][-1],
    "final negative fraction": result["negative_coordinate_fraction"][-1],
}])
display(timing_summary.T.rename(columns={0: "value"}))

In [ ]:
# BLOCK 5 — Measure and group the raw endpoints at T_FINAL.
dtb_endpoints = result["trajectory"][-1]
reference_endpoints = reference_trajectory[-1]

# Read-only residual diagnostics on the actual particle coordinates.
dtb_endpoint_drift = torch.linalg.vector_norm(
    cournot_velocity(dtb_endpoints.to(DEVICE)), dim=1
).cpu()
reference_endpoint_drift = torch.linalg.vector_norm(
    cournot_velocity(reference_endpoints.to(DEVICE)), dim=1
).cpu()

known_cpu = known_equilibria.cpu()
dtb_known_distance, dtb_known_index = torch.cdist(
    dtb_endpoints, known_cpu
).min(dim=1)
reference_known_distance, reference_known_index = torch.cdist(
    reference_endpoints, known_cpu
).min(dim=1)

def dominant_pair_codes(points):
    top_two = points.topk(2, dim=1).indices.sort(dim=1).values
    return top_two[:, 0] * DIM + top_two[:, 1], top_two

# Dominant coordinates label finite-time groups, not established basins.
pair_codes, top_two_coordinates = dominant_pair_codes(dtb_endpoints)
candidate_rows = []
for code in pair_codes.unique(sorted=True):
    member_mask = pair_codes == code
    members = dtb_endpoints[member_mask]
    candidate = members.mean(dim=0)
    first, second = divmod(int(code), DIM)
    exact_pair = torch.zeros(DIM)
    exact_pair[first] = exact_pair[second] = 0.5
    candidate_rows.append({
        "dominant pair": f"({first + 1},{second + 1})",
        "particles": int(member_mask.sum()),
        "occupancy": float(member_mask.float().mean()),
        "mean endpoint drift norm": float(dtb_endpoint_drift[member_mask].mean()),
        "group-center drift norm": float(torch.linalg.vector_norm(
            cournot_velocity(candidate.to(DEVICE)).cpu()
        )),
        "group-center distance to exact pair equilibrium": float(torch.linalg.vector_norm(
            candidate - exact_pair
        )),
        "mean largest coordinate": float(members.max(dim=1).values.mean()),
    })
candidate_table = pd.DataFrame(candidate_rows).sort_values(
    "particles", ascending=False
).reset_index(drop=True)

assignment_counts = torch.bincount(
    dtb_known_index, minlength=len(known_equilibrium_names)
)
assignment_table = pd.DataFrame({
    "nearest known equilibrium": known_equilibrium_names,
    "assigned particles": assignment_counts.numpy(),
    "fraction": (assignment_counts / N_PARTICLES).numpy(),
}).sort_values("assigned particles", ascending=False).reset_index(drop=True)

print({
    "endpoint_time": N_STEPS * H,
    "DTB_endpoint_drift_RMS": float(dtb_endpoint_drift.square().mean().sqrt()),
    "Euler_endpoint_drift_RMS": float(reference_endpoint_drift.square().mean().sqrt()),
    "DTB_median_distance_to_known_equilibrium": float(dtb_known_distance.median()),
    "DTB_p90_distance_to_known_equilibrium": float(torch.quantile(dtb_known_distance, 0.9)),
    "observed_dominant_pairs": len(candidate_table),
})
display(candidate_table.head(15))
display(assignment_table.head(15))

In [ ]:
# BLOCK 6 — Fixed-basis PCA views of the high-dimensional pushforward.
# Fit one projection basis and reuse it for every method and time.
pca_fit = torch.cat((
    result["trajectory"][0], result["trajectory"][-1],
    reference_trajectory[-1], known_cpu,
), dim=0)
pca_center = pca_fit.mean(dim=0)
_, pca_singular_values, pca_vh = torch.linalg.svd(
    pca_fit - pca_center, full_matrices=False
)
pca_basis = pca_vh[:2].T
explained_fraction = (
    pca_singular_values[:2].square().sum()
    / pca_singular_values.square().sum()
)

def project_pca(points):
    return (points - pca_center) @ pca_basis

dominant_coordinate = dtb_endpoints.argmax(dim=1).numpy()
known_pca = project_pca(known_cpu).numpy()
fig, axes = plt.subplots(
    2, len(SNAPSHOT_STEPS), figsize=(3.1 * len(SNAPSHOT_STEPS), 6),
    sharex=True, sharey=True
)
for column, step in enumerate(SNAPSHOT_STEPS):
    dtb_projection = project_pca(result["trajectory"][step]).numpy()
    euler_projection = project_pca(reference_trajectory[step]).numpy()
    axes[0, column].scatter(
        dtb_projection[:, 0], dtb_projection[:, 1], c=dominant_coordinate,
        cmap="tab10", vmin=-0.5, vmax=DIM - 0.5, s=9, alpha=.65
    )
    axes[1, column].scatter(
        euler_projection[:, 0], euler_projection[:, 1], c=dominant_coordinate,
        cmap="tab10", vmin=-0.5, vmax=DIM - 0.5, s=9, alpha=.65
    )
    for row in range(2):
        axes[row, column].scatter(
            known_pca[:, 0], known_pca[:, 1], marker="x", c="black", s=18, alpha=.35
        )
        axes[row, column].grid(alpha=.2)
    axes[0, column].set_title(f"t={step * H:.2f}")
axes[0, 0].set_ylabel("DTB pushforward: PC2")
axes[1, 0].set_ylabel("Euler reference: PC2")
for axis in axes[1]:
    axis.set_xlabel("PC1")
plt.suptitle(
    f"Fixed PCA projection; first two PCs show {100 * float(explained_fraction):.1f}% "
    "of fitted variance",
    y=1.02
)
plt.tight_layout(); plt.show()

print(
    "PCA warning: proximity or separation in this figure is descriptive only; "
    f"{100 * (1 - float(explained_fraction)):.1f}% of fitted variance is hidden."
)

In [ ]:
# BLOCK 7 — Coordinate, covariance, parallel-coordinate, and endpoint views.
snapshot_tensor = result["trajectory"][SNAPSHOT_STEPS]
reference_snapshot_tensor = reference_trajectory[SNAPSHOT_STEPS]

fig, axes = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True)
for row, (values, title) in enumerate((
    (snapshot_tensor, "DTB"), (reference_snapshot_tensor, "Euler reference")
)):
    means = values.mean(dim=1).numpy().T
    standard_deviations = values.std(dim=1).numpy().T
    image_mean = axes[row, 0].imshow(means, aspect="auto", cmap="viridis")
    image_std = axes[row, 1].imshow(standard_deviations, aspect="auto", cmap="magma")
    axes[row, 0].set_title(f"{title}: coordinate means")
    axes[row, 1].set_title(f"{title}: coordinate standard deviations")
    for axis in axes[row]:
        axis.set_yticks(range(DIM), [f"x{i + 1}" for i in range(DIM)])
        axis.set_xticks(range(len(SNAPSHOT_STEPS)),
                        [f"{step * H:.2f}" for step in SNAPSHOT_STEPS])
        axis.set_xlabel("time")
    plt.colorbar(image_mean, ax=axes[row, 0], shrink=.8)
    plt.colorbar(image_std, ax=axes[row, 1], shrink=.8)
plt.show()

def correlation_matrix(points):
    centered = points - points.mean(dim=0)
    covariance = centered.T @ centered / max(1, points.shape[0] - 1)
    scales = torch.sqrt(torch.diagonal(covariance).clamp_min(1e-15))
    return covariance / (scales[:, None] * scales[None, :])

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
for axis, points, title in (
    (axes[0], result["trajectory"][0], "initial"),
    (axes[1], result["trajectory"][-1], "DTB final"),
    (axes[2], reference_trajectory[-1], "Euler final"),
):
    image = axis.imshow(correlation_matrix(points), vmin=-1, vmax=1, cmap="coolwarm")
    axis.set_title(f"coordinate correlation: {title}")
    axis.set_xticks(range(DIM)); axis.set_yticks(range(DIM))
    axis.set_xticklabels(range(1, DIM + 1)); axis.set_yticklabels(range(1, DIM + 1))
plt.colorbar(image, ax=axes, shrink=.8); plt.show()

# Parallel coordinates retain all ten coordinates for a reproducible particle subset.
subset = torch.linspace(0, N_PARTICLES - 1, min(80, N_PARTICLES)).long()
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for axis, points, title in (
    (axes[0], dtb_endpoints[subset], "DTB endpoint"),
    (axes[1], reference_endpoints[subset], "Euler reference endpoint"),
):
    for local_index, particle in enumerate(points):
        original_index = int(subset[local_index])
        axis.plot(range(1, DIM + 1), particle.numpy(), alpha=.28,
                  color=plt.cm.tab10(dominant_coordinate[original_index] / max(1, DIM - 1)))
    axis.set_title(title); axis.set_xlabel("coordinate")
    axis.set_xticks(range(1, DIM + 1)); axis.grid(alpha=.2)
axes[0].set_ylabel("quantity")
plt.tight_layout(); plt.show()

# Count the two dominant coordinates of each raw DTB endpoint.
occupancy_matrix = np.zeros((DIM, DIM), dtype=int)
for pair in top_two_coordinates.numpy():
    occupancy_matrix[pair[0], pair[1]] += 1
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=True)
masked_occupancy = np.ma.masked_where(occupancy_matrix == 0, occupancy_matrix)
image = axes[0].imshow(masked_occupancy, cmap="viridis")
axes[0].set(title="DTB endpoint dominant-pair counts", xlabel="second dominant coordinate",
            ylabel="first dominant coordinate")
axes[0].set_xticks(range(DIM), range(1, DIM + 1)); axes[0].set_yticks(range(DIM), range(1, DIM + 1))
plt.colorbar(image, ax=axes[0])

top_candidates = candidate_table.head(min(12, len(candidate_table)))
axes[1].barh(top_candidates["dominant pair"][::-1], top_candidates["particles"][::-1])
axes[1].set(title="largest DTB endpoint groups", xlabel="particles")
axes[2].hist(dtb_known_distance.numpy(), bins=30, alpha=.65, label="DTB endpoint")
axes[2].hist(reference_known_distance.numpy(), bins=30, alpha=.55, label="Euler endpoint")
axes[2].set(title="endpoint distances at T_FINAL", xlabel="distance to nearest known equilibrium",
            ylabel="particles")
axes[2].legend(); axes[2].grid(alpha=.2)
plt.show()

In [ ]:
# BLOCK 8 — Accuracy, equilibrium approach, and where the runtime is spent.
state_times = np.arange(N_STEPS + 1) * H
step_times = np.arange(1, N_STEPS + 1) * H
fig, axes = plt.subplots(2, 3, figsize=(16, 8), constrained_layout=True)

axes[0, 0].semilogy(state_times, trajectory_rmse + 1e-15, label="absolute")
axes[0, 0].semilogy(state_times, relative_rmse + 1e-15, label="relative")
axes[0, 0].set_title("DTB particle error against Euler"); axes[0, 0].legend()
axes[0, 1].semilogy(step_times, result["projection_residual"] + 1e-15)
axes[0, 1].set_title("tangent projection residual")
axes[0, 2].semilogy(step_times, result["target_drift_rms"] + 1e-15, label="DTB")
axes[0, 2].semilogy(state_times, reference_drift_rms + 1e-15, label="Euler")
axes[0, 2].set_title("RMS game velocity"); axes[0, 2].legend()

axes[1, 0].plot(step_times, result["solve_seconds"], label="cached coefficient solve")
axes[1, 0].plot(step_times, result["update_seconds"], label="update/diagnostics")
axes[1, 0].set_title("frozen-basis seconds per step"); axes[1, 0].legend()
axes[1, 1].semilogy(step_times, result["nearest_known_median"] + 1e-15,
                     label="median")
axes[1, 1].semilogy(step_times, result["nearest_known_p90"] + 1e-15,
                     label="90th percentile")
axes[1, 1].set_title("distance to analytic equilibrium set"); axes[1, 1].legend()
axes[1, 2].plot(step_times, result["negative_coordinate_fraction"],
                 label="negative coordinates")
axes[1, 2].plot(step_times, result["retained_rank"] / selected.numel(),
                 label="retained rank / basis size")
axes[1, 2].set_title("constraint and numerical health"); axes[1, 2].legend()
for axis in axes.ravel():
    axis.set_xlabel("time"); axis.grid(alpha=.25)
plt.show()

total_particle_coordinate_updates = N_STEPS * N_PARTICLES * DIM
print({
    "end_to_end_DTB_steps_per_second": N_STEPS / result["wall_seconds"],
    "integration_only_steps_per_second": N_STEPS / result["integration_seconds"],
    "end_to_end_particle_coordinates_per_second": total_particle_coordinate_updates / result["wall_seconds"],
    "fraction_time_in_one_time_basis_setup": result["basis_setup_seconds"] / result["wall_seconds"],
    "fraction_time_in_one_time_SVD_setup": result["svd_setup_seconds"] / result["wall_seconds"],
    "fraction_time_in_cached_coefficient_solves": float(np.sum(result["solve_seconds"]) / result["wall_seconds"]),
    "final_DTB_drift_RMS": float(dtb_endpoint_drift.square().mean().sqrt()),
    "final_Euler_drift_RMS": float(reference_drift_rms[-1]),
})

In [ ]:
# BLOCK 9 — Optional one-step dimension-scaling benchmark.
def benchmark_one_dimension(dim):
    torch.manual_seed(SEED + dim)
    local_labels = sample_simplex(BENCHMARK_PARTICLES, dim, INITIAL_TOTAL)
    local_model = ResidualMLPMap(
        dim=dim, width=MLP_WIDTH, depth=MLP_DEPTH,
        activation="tanh", dtype=DTYPE
    ).to(DEVICE)
    local_theta, local_structure, _ = flat_params(local_model)
    local_theta = local_theta.to(DEVICE)
    local_selected, local_forced, local_total = choose_tangent_parameters(
        local_structure, MLP_DEPTH, BENCHMARK_BASIS_SIZE
    )

    synchronize(); start = time.perf_counter()
    local_x, local_jacobians, local_stacked = game_dtb_basis_matrices(
        local_theta, local_selected, local_labels, local_model,
        local_structure, chunk=min(JACOBIAN_CHUNK, BENCHMARK_PARTICLES)
    )
    synchronize(); after_basis = time.perf_counter()
    _, local_svd = truncated_svd_solve(
        local_stacked, cournot_velocity(local_x).reshape(-1), rtol=SVD_RTOL
    )
    synchronize(); after_solve = time.perf_counter()
    return {
        "dimension": dim,
        "particles": BENCHMARK_PARTICLES,
        "total parameters": local_total,
        "selected basis": int(local_selected.numel()),
        "forced output params": local_forced,
        "system rows": BENCHMARK_PARTICLES * dim,
        "J storage MB": local_stacked.numel() * local_stacked.element_size() / 2**20,
        "basis seconds": after_basis - start,
        "SVD seconds": after_solve - after_basis,
        "retained rank": local_svd.retained_rank,
        "projection residual": local_svd.relative_residual,
    }

if RUN_SCALING_BENCHMARK:
    _ = benchmark_one_dimension(BENCHMARK_DIMS[0])  # warm up kernels
    scaling_table = pd.DataFrame([
        benchmark_one_dimension(dim) for dim in BENCHMARK_DIMS
    ])
    display(scaling_table)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
    axes[0].plot(scaling_table["dimension"], scaling_table["basis seconds"], "o-",
                 label="Jacobian basis")
    axes[0].plot(scaling_table["dimension"], scaling_table["SVD seconds"], "o-",
                 label="truncated SVD")
    axes[0].set(title="one-step wall time", ylabel="seconds"); axes[0].legend()
    axes[1].plot(scaling_table["dimension"], scaling_table["J storage MB"], "o-")
    axes[1].set(title="explicit tangent matrix storage", ylabel="MiB")
    axes[2].plot(scaling_table["dimension"], scaling_table["projection residual"], "o-")
    axes[2].set(title="initial projection residual", ylabel="relative residual")
    for axis in axes:
        axis.set_xlabel("game dimension"); axis.grid(alpha=.25)
    plt.show()
else:
    print("Dimension-scaling benchmark skipped; set RUN_SCALING_BENCHMARK=True to run it.")

## How to review the 10D boundary experiment

### What the notebook can establish

- **Approximation quality:** `trajectory_rmse` and `projection_residual` distinguish accumulated trajectory error from instantaneous tangent-space error.
- **Computational boundary:** synchronized one-time `basis_setup_seconds` and `svd_setup_seconds`, cached per-step `solve_seconds`, explicit Jacobian storage, and the dimension sweep separate initialization cost from particle-integration cost.
- **Equilibrium evidence:** evaluate game-velocity residuals and distances to known equilibria directly on the DTB endpoints at `T_FINAL`. Groups defined by the two dominant coordinates summarize finite-time particle concentration; their centers and individual endpoint residuals are reported separately.
- **Pushforward structure:** fixed-basis PCA shows global movement, while coordinate heatmaps, correlations, parallel coordinates, and the dominant-pair count matrix retain information that PCA hides.

### What counts as success

DTB is useful in this experiment when its particles remain feasible, the projection residual stays controlled, and the DTB/Euler discrepancy is acceptable for the intended use. Claims of equilibrium approach require small residuals and small distances measured at the actual DTB endpoints. Low drift norms and small direct equilibrium residuals matter more than visually tight clusters in PCA.

### Warning signs at the boundary

- A rapidly increasing projection residual means the selected tangent space cannot represent the 10D game field.
- Small retained rank together with large `alpha_norm` signals an ill-conditioned tangent coefficient solve.
- Because \(\theta\) is frozen, a rising residual means the original tangent span no longer represents the velocity field well along the transported particles. This is the intended limitation under test, not a hidden network update.
- A growing negative-coordinate fraction means the unconstrained tangent projection is violating the quantity domain even though the target field points inward.
- PCA may merge distinct equilibria or make distant points appear close; always compare its explained-variance fraction with the full-coordinate diagnostics.
- Setup runtime and memory scale with the explicit matrix shape `(N_PARTICLES * DIM, BASIS_SIZE)`. This implementation is not matrix-free, so the one-time SVD can become the limiting component even though later particle steps reuse it.

### Equilibrium interpretation

For `DIM=10` and `mu=2`, the notebook validates the origin, one symmetric interior point with coordinate \(17/162\), and all \(\binom{10}{2}=45\) pair-support points with two coordinates equal to \(1/2\). These are symmetry-generated candidates, not a proof that the list is exhaustive. Boundary equilibria are nonsmooth points of the best-response field, so a conventional unconstrained Jacobian stability test can be misleading; the notebook reports finite-time trajectories, endpoint feasibility, and direct residuals.

All endpoint diagnostics read the raw final particles from the ordinary DTB tangent-bundle update. Negative coordinates remain visible in the results. Nearest-equilibrium assignments and dominant-coordinate groups are descriptive and do not establish convergence or basin membership. To examine longer-time behavior, increase `T_FINAL` so that the same DTB tangent-bundle update performs every additional step; the separate Euler reference uses the same horizon. Failure to approach an equilibrium is not proof that it does not exist, because finite particles, finite time, and the restricted tangent basis can all miss it.

To test dimensions above ten, first inspect the scaling table, then change `DIM`, `N_PARTICLES`, and `BASIS_SIZE` together. Holding a small basis fixed while increasing dimension tests representation failure; increasing the basis tests the corresponding runtime and memory cost.